In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/kaggriculture/README.md
/kaggle/input/competitions/kaggriculture/AGENTS.md


In [2]:
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds

# =====================================================================
# 1. SOLVER DE ALOCAÇÃO DE TRABALHADORES (MILP)
# =====================================================================
class FarmWorkerAllocator:
    def optimize_turn_actions(self, total_workers, pending_tasks):
        if not pending_tasks or total_workers <= 0:
            return {'selected_tasks': [], 'workers_used': 0}

        num_tasks = len(pending_tasks)
        
        # Minimizar: Sum(-1 * Valor * Urgencia * x_i)
        c = -np.array([task['value'] * task.get('urgency_weight', 1.0) for task in pending_tasks], dtype=float)
        
        # Restrição: Sum(Custo_i * x_i) <= total_workers
        worker_costs = np.array([task['cost'] for task in pending_tasks], dtype=float)
        constraints = LinearConstraint(worker_costs, lb=0, ub=total_workers)

        # Variáveis inteiras com limites superiores por tarefa
        integrality = np.ones(num_tasks, dtype=int)
        ub_bounds = [min(task.get('max_instances', 1), total_workers) for task in pending_tasks]
        bounds = Bounds(lb=0, ub=ub_bounds)

        res = milp(c=c, integrality=integrality, constraints=constraints, bounds=bounds)

        if not res.success:
            return {'selected_tasks': [], 'workers_used': 0}

        solution_counts = np.round(res.x).astype(int)
        selected_tasks = []
        workers_used = 0

        for idx, count in enumerate(solution_counts):
            if count > 0:
                task_info = pending_tasks[idx]
                workers_used += task_info['cost'] * count
                selected_tasks.append({
                    'action': task_info['action'],
                    'cost': task_info['cost'] * count,
                    'count': count
                })

        return {'selected_tasks': selected_tasks, 'workers_used': workers_used}


# =====================================================================
# 2. SIMULADOR LOCAL DO AMBIENTE (720 TURNOS)
# =====================================================================
class LocalFarmSimulator:
    def __init__(self, initial_cash=500, total_turns=720):
        self.total_turns = total_turns
        self.current_turn = 1
        self.base_prices = {
            'wheat': 15.0, 'carrot': 20.0, 'melon': 100.0, 
            'tomato': 40.0, 'strawberry': 60.0,
            'eggs': 25.0, 'milk': 50.0, 'wool': 80.0
        }
        self.market_prices = self.base_prices.copy()
        self.state = {
            'cash': initial_cash,
            'workers': 3,
            'worker_cost_per_turn': 5,
            'tiles': [{'id': i, 'crop': None, 'age': 0, 'watered': False, 'fertilized': False, 'harvests': 0} for i in range(12)],
            'animals': [{'type': 'cow', 'fed': False, 'held': 0, 'interval': 3, 'max_held': 5}],
            'inventory': {k: 0 for k in self.base_prices.keys()},
            'fertilizer_count': 10
        }

    def reset(self):
        self.__init__()
        return self._get_observation()

    def _get_observation(self):
        return {
            'turn': self.current_turn,
            'turns_remaining': self.total_turns - self.current_turn,
            'cash': self.state['cash'],
            'workers': self.state['workers'],
            'inventory': self.state['inventory'].copy(),
            'market_prices': self.market_prices.copy(),
            'tiles': [t.copy() for t in self.state['tiles']],
            'animals': [a.copy() for a in self.state['animals']],
            'fertilizer_count': self.state['fertilizer_count']
        }

    def step(self, agent_actions):
        if self.current_turn > self.total_turns:
            raise RuntimeError("A simulação já foi encerrada.")

        # 1. Custo de manutenção de trabalhadores
        self.state['cash'] -= self.state['workers'] * self.state['worker_cost_per_turn']

        # 2. Execução de ações do agente
        for action in agent_actions:
            act_type = action.get('type')
            
            if act_type == 'sell':
                item = action['item']
                qty = min(action['qty'], self.state['inventory'].get(item, 0))
                if qty > 0:
                    unit_price = self.market_prices[item]
                    self.state['cash'] += qty * unit_price
                    self.state['inventory'][item] -= qty
                    # Impacto de mercado
                    self.market_prices[item] = max(5.0, self.market_prices[item] * np.exp(-0.015 * qty))

            elif act_type == 'plant':
                tile = self.state['tiles'][action['tile_id']]
                if tile['crop'] is None:
                    tile['crop'] = action['crop']
                    tile['age'] = 0
                    tile['watered'] = True
                    tile['fertilized'] = False
                    tile['harvests'] = 0

            elif act_type == 'irrigate':
                tile = self.state['tiles'][action['tile_id']]
                if tile['crop'] is not None:
                    tile['watered'] = True

            elif act_type == 'fertilize':
                tile = self.state['tiles'][action['tile_id']]
                if tile['crop'] is not None and not tile['fertilized'] and self.state['fertilizer_count'] > 0:
                    tile['fertilized'] = True
                    self.state['fertilizer_count'] -= 1

            elif act_type == 'harvest':
                tile = self.state['tiles'][action['tile_id']]
                if tile['crop'] is not None:
                    yield_qty = self._calculate_yield(tile)
                    if yield_qty > 0:
                        self.state['inventory'][tile['crop']] += yield_qty
                        tile['harvests'] += 1
                        if tile['crop'] in ['wheat', 'carrot', 'melon']:
                            tile['crop'] = None
                        elif tile['crop'] in ['tomato', 'strawberry'] and tile['harvests'] >= 4:
                            tile['crop'] = 'weed'

            elif act_type == 'clear_weed':
                tile = self.state['tiles'][action['tile_id']]
                if tile['crop'] == 'weed':
                    tile['crop'] = None

            elif act_type == 'feed_animal':
                for anim in self.state['animals']:
                    if not anim['fed']:
                        anim['fed'] = True
                        break

        # 3. Atualização biológica do turno
        self._update_biology()

        # 4. Recuperação gradual do mercado
        for item in self.market_prices:
            self.market_prices[item] += (self.base_prices[item] - self.market_prices[item]) * 0.05

        self.current_turn += 1
        done = self.current_turn > self.total_turns
        return self._get_observation(), self.state['cash'], done

    def _calculate_yield(self, tile):
        crop, age, watered, fertilized = tile['crop'], tile['age'], tile['watered'], tile['fertilized']
        if not watered: return 1
        if crop == 'wheat': return 6 if fertilized else 4
        if crop == 'carrot': return 4 if fertilized else 3
        if crop == 'melon': return (1 + min(age, 10)) + (2 if fertilized else 0) if 6 <= age <= 12 else 2
        if crop in ['tomato', 'strawberry']: return 3 if fertilized else 2
        return 1

    def _update_biology(self):
        for tile in self.state['tiles']:
            if tile['crop'] is not None:
                tile['age'] += 1
                tile['watered'] = False
        for anim in self.state['animals']:
            if anim['fed']:
                if self.current_turn % anim['interval'] == 0 and anim['held'] < anim['max_held']:
                    anim['held'] += 1
                    self.state['inventory']['milk'] += 1
                anim['fed'] = False


# =====================================================================
# 3. AGENTE INTEGRADO (GERA TAREFAS + OTIMIZA ALOCAÇÃO)
# =====================================================================
class IntegratedFarmAgent:
    def __init__(self):
        self.allocator = FarmWorkerAllocator()

    def act(self, obs):
        actions = []
        pending_tasks = []
        
        # --- A. TRATAMENTO DE ESTOQUE (Drip-Selling sem gastar mão de obra) ---
        for item, qty in obs['inventory'].items():
            if qty > 0:
                drip_qty = max(1, min(qty, 3)) # Vende no máximo 3 unidades por turno para conter impacto
                actions.append({'type': 'sell', 'item': item, 'qty': drip_qty})

        # --- B. GERAR LISTA DE TAREFAS CANDIDATAS QUE EXIGEM TRABALHADORES ---
        
        # 1. Alimentar animais sem ração
        for anim in obs['animals']:
            if not anim['fed']:
                pending_tasks.append({
                    'id': 'feed_cow',
                    'action': {'type': 'feed_animal'},
                    'cost': 1, 'value': 50.0, 'urgency_weight': 2.5
                })

        # 2. Varrer terrenos
        for tile in obs['tiles']:
            t_id = tile['id']
            crop = tile['crop']
            
            # Limpar ervas daninhas
            if crop == 'weed':
                pending_tasks.append({
                    'id': f'clear_{t_id}',
                    'action': {'type': 'clear_weed', 'tile_id': t_id},
                    'cost': 1, 'value': 80.0, 'urgency_weight': 3.0
                })

            # Plantar se o terreno estiver vazio
            elif crop is None:
                # Escolhe Melão se estiver longe do fim, senão Cenoura/Trigo
                selected_crop = 'melon' if obs['turns_remaining'] > 30 else 'carrot'
                pending_tasks.append({
                    'id': f'plant_{t_id}',
                    'action': {'type': 'plant', 'tile_id': t_id, 'crop': selected_crop},
                    'cost': 1, 'value': 70.0, 'urgency_weight': 1.5
                })

            # Irrigate / Harvest se houver cultivo
            else:
                # Rega obrigatória se ainda não foi regado no turno
                if not tile['watered']:
                    urgency = 2.0 if crop == 'melon' and 6 <= tile['age'] <= 10 else 1.2
                    pending_tasks.append({
                        'id': f'water_{t_id}',
                        'action': {'type': 'irrigate', 'tile_id': t_id},
                        'cost': 1, 'value': 40.0, 'urgency_weight': urgency
                    })

                # Adubar melão jovem
                if not tile['fertilized'] and crop == 'melon' and tile['age'] <= 5 and obs['fertilizer_count'] > 0:
                    pending_tasks.append({
                        'id': f'fert_{t_id}',
                        'action': {'type': 'fertilize', 'tile_id': t_id},
                        'cost': 1, 'value': 60.0, 'urgency_weight': 1.8
                    })

                # Colheita na idade de pico
                if crop == 'melon' and tile['age'] >= 10:
                    pending_tasks.append({
                        'id': f'harvest_{t_id}',
                        'action': {'type': 'harvest', 'tile_id': t_id},
                        'cost': 1, 'value': 150.0, 'urgency_weight': 2.5
                    })
                elif crop in ['wheat', 'carrot'] and tile['age'] >= 3:
                    pending_tasks.append({
                        'id': f'harvest_{t_id}',
                        'action': {'type': 'harvest', 'tile_id': t_id},
                        'cost': 1, 'value': 50.0, 'urgency_weight': 1.5
                    })

        # --- C. SOLVER DE OTIMIZAÇÃO DE MÃO DE OBRA ---
        allocation = self.allocator.optimize_turn_actions(obs['workers'], pending_tasks)
        
        # Adiciona ações selecionadas pelo solver ao plano do turno
        for task in allocation['selected_tasks']:
            actions.append(task['action'])

        return actions


# =====================================================================
# 4. EXECUÇÃO DO TESTE COMPLETO DE 720 TURNOS
# =====================================================================
if __name__ == "__main__":
    env = LocalFarmSimulator(initial_cash=500, total_turns=720)
    agent = IntegratedFarmAgent()
    
    obs = env.reset()
    done = False
    
    print("=========================================================")
    print(" INICIANDO SIMULAÇÃO COMPLETA DE 720 TURNOS (30 DIAS)")
    print("=========================================================")
    
    while not done:
        actions = agent.act(obs)
        obs, cash, done = env.step(actions)

        # Telemetria a cada 120 turnos (5 dias de jogo)
        if obs['turn'] % 120 == 0 or done:
            day = obs['turn'] // 24
            print(f"Dia {day:2d} (Turno {obs['turn']:3d}/720) | Caixa: ${cash:8.2f} | Trabalhadores: {obs['workers']} | Fertilizante: {obs['fertilizer_count']}")

    print("\n=========================================================")
    print(" SIMULAÇÃO CONCLUÍDA COM SUCESSO!")
    print("=========================================================")
    print(f" Saldo Final Obtido: ${cash:.2f}")
    print(f" Estoque Final: {obs['inventory']}")

 INICIANDO SIMULAÇÃO COMPLETA DE 720 TURNOS (30 DIAS)
Dia  5 (Turno 120/720) | Caixa: $ 7784.88 | Trabalhadores: 3 | Fertilizante: 0
Dia 10 (Turno 240/720) | Caixa: $15405.75 | Trabalhadores: 3 | Fertilizante: 0
Dia 15 (Turno 360/720) | Caixa: $23026.35 | Trabalhadores: 3 | Fertilizante: 0
Dia 20 (Turno 480/720) | Caixa: $30646.94 | Trabalhadores: 3 | Fertilizante: 0
Dia 25 (Turno 600/720) | Caixa: $38267.54 | Trabalhadores: 3 | Fertilizante: 0
Dia 30 (Turno 720/720) | Caixa: $44805.09 | Trabalhadores: 3 | Fertilizante: 0
Dia 30 (Turno 721/720) | Caixa: $44790.09 | Trabalhadores: 3 | Fertilizante: 0

 SIMULAÇÃO CONCLUÍDA COM SUCESSO!
 Saldo Final Obtido: $44790.09
 Estoque Final: {'wheat': 0, 'carrot': 2, 'melon': 0, 'tomato': 0, 'strawberry': 0, 'eggs': 0, 'milk': 0, 'wool': 0}
